# 第 0 階段: 複利與等比級數 (Compound Interest & Geometric Series)

**目標**
1. 理解單筆投入 (lump sum) 的複利 (compound interest) 公式, 以及年利率怎麼換成月利率
2. 用**等比級數 (geometric series)**推導 "每月定期定額 (regular monthly contribution)" 的終值 (future value, FV) 公式
3. 用公式驗證 `assetsim.cashflow.simulate()` 的逐月模擬是對的
4. 看參數 (報酬率, 生活開銷, 年數) 對結果的影響

> 這一階段假設報酬率**固定不變**(deterministic). 真實市場每年報酬都在波動 (volatility), 那是第 1 階段 (蒙地卡羅, Monte Carlo) 要處理的.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from assetsim.cashflow import Asset, Params, monthly_rate, simulate
from assetsim.formulas import fv_annuity, fv_lump_sum
from assetsim.plotting import setup_cjk_font

setup_cjk_font()

## 1. 單筆投入: 複利 (lump sum & compound interest)

本金 $P$, 每期報酬率 $i$, 經過 1 期變成 $P(1+i)$, 2 期變成 $P(1+i)^2$......$n$ 期後:

$$FV = P(1+i)^n$$

這是**指數成長 (exponential growth)**: 每一期的增加量和當下的金額成正比.

**72 法則 (rule of 72)**: 翻倍時間 (doubling time) 約為 $72 / (100 \cdot i)$ 期. 推導: 解 $(1+i)^n = 2$ 得 $n = \dfrac{\ln 2}{\ln(1+i)}$, 而 $i$ 很小時 $\ln(1+i) \approx i$(泰勒展開 (Taylor expansion) 的一階項, first-order term), 所以 $n \approx 0.693 / i$. 用 72 而不是 69.3, 是因為 72 的因數多, 心算方便, 而且在 8% 附近剛好更準.

In [ ]:
for r in [0.04, 0.07, 0.10]:
    exact = np.log(2) / np.log(1 + r)
    print(f"年報酬 {r:.0%}: 精確翻倍 {exact:.2f} 年, 72 法則 {72 / (r * 100):.2f} 年")

## 2. 年利率 → 月利率 (annual → monthly rate)

模擬是逐月進行, 需要月報酬率. 常見兩種做法:

- **單利換算 (simple division)**: $i = r / 12$
- **等效換算 (equivalent rate)**: $i = (1+r)^{1/12} - 1$, 保證 $(1+i)^{12} = 1+r$

基金, ETF 公布的 "年化報酬率" 是複利後的結果, 所以應該用**等效換算**. 用 $r/12$ 會高估, 因為 $(1 + r/12)^{12} > 1 + r$(這就是 "名目利率 (nominal rate)" 和 "有效利率 (effective rate)" 的差別).

In [ ]:
r = 0.07
i_simple, i_equiv = r / 12, monthly_rate(r)
print(f"r/12 = {i_simple:.6f}, 一年後實際變成 {(1 + i_simple) ** 12 - 1:.4%}")
print(f"等效 = {i_equiv:.6f}, 一年後實際變成 {(1 + i_equiv) ** 12 - 1:.4%}")
print(f"30 年後差距: {((1 + i_simple) ** 360 / (1 + i_equiv) ** 360 - 1):.1%}")

## 3. 定期定額: 等比級數 (annuity & geometric series)

每月**月底**投入 $C$, 月報酬率 $i$, 共 $n$ 個月. 令公比 (common ratio)$q = 1+i$:

- 第 1 個月投入的錢, 複利 $n-1$ 期 → $Cq^{n-1}$
- 第 2 個月投入的錢 → $Cq^{n-2}$
- ......
- 最後一個月投入的錢, 還沒成長 → $C$

$$S = C(1 + q + q^2 + \cdots + q^{n-1})$$

**求和技巧 (sum of a geometric series)**: 兩邊乘以 $q$, 再相減, 中間項全部消掉:

$$qS - S = C(q^n - 1) \quad\Rightarrow\quad S = C\,\frac{q^n - 1}{q - 1} = C\,\frac{(1+i)^n - 1}{i}$$

這個 "乘以公比再相減" 的技巧, 之後在機率 (幾何分布 geometric distribution 的期望值 expected value) 和強化學習 (reinforcement learning; 折扣報酬 discounted return $\sum \gamma^t r_t$) 都會再出現.

In [ ]:
C, n = 18_000, 30 * 12
i = monthly_rate(0.07)

brute = sum(C * (1 + i) ** (n - k) for k in range(1, n + 1))  # 一項一項加 (brute-force sum)
print(f"逐項加總: {brute:,.0f}")
print(f"公式:    {fv_annuity(C, i, n):,.0f}")

## 4. 用公式驗證模擬器

`simulate()` 是逐月迴圈, `formulas.py` 是封閉解 (closed-form solution). 在兩者都適用的情況下 (單一資產, 不加薪, 開銷固定, 沒有年終, 緊急預備金, 股息和費用), 結果應該一模一樣.

一般情況下的總資產 = 起始資產的複利 + 每月投入的年金終值 (future value of an annuity):

$$FV = P(1+i)^n + C\,\frac{(1+i)^n - 1}{i}$$

(`tests/test_cashflow.py` 裡也有同樣的檢查.)

In [ ]:
# 每月結餘 = 60,000 - 42,000 = 18,000; Asset 只設價格報酬, 股息和費用都是 0
p = Params(initial_assets=1_000_000, monthly_salary=60_000, annual_raise=0, bonus_months=0,
           monthly_expenses=42_000, inflation=0, emergency_months=0,
           assets={"ETF": Asset(1.0, price_return=0.07)}, years=30, rebalance=False)
sim = simulate(p)["total"].iloc[-1]
formula = fv_lump_sum(1_000_000, i, n) + fv_annuity(18_000, i, n)
print(f"模擬: {sim:,.0f}\n公式: {formula:,.0f}\n差距: {sim - formula:.2e}")

## 5. 複利的雪球: 本金 vs 收益 (principal vs gain)

用預設參數 (見 `Params`: ETF 60% / 股票 30% / 現金 10%, 含股息, 內扣費用, 手續費, 年終與緊急預備金, 每年再平衡 rebalancing). 注意收益 (兩條線之間) 在後期才明顯拉開.

In [ ]:
df = simulate(Params())
yearly = df[df["month"] % 12 == 0]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4.5))
ax1.fill_between(yearly["year"], 0, yearly["invested"] / 1e6, label="投入本金", alpha=0.7)
ax1.fill_between(yearly["year"], yearly["invested"] / 1e6, yearly["total"] / 1e6, label="投資收益", alpha=0.7)
ax1.set(xlabel="年", ylabel="百萬 TWD", title="總資產組成")
ax1.legend()

ax2.plot(yearly["year"], yearly["gain"] / yearly["total"])
ax2.set(xlabel="年", ylabel="比例", title="收益佔總資產的比例")
ax2.yaxis.set_major_formatter(plt.matplotlib.ticker.PercentFormatter(1.0))
plt.tight_layout()

## 6. 敏感度分析 (sensitivity analysis): 哪個參數影響最大?

固定其他參數, 一次只改一個, 看 30 年後的總資產.

In [ ]:
def final_total(**kw):
    return simulate(Params(**kw))["total"].iloc[-1] / 1e6

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

rates = np.arange(0.02, 0.121, 0.01)
def with_price_return(r):
    """預設資產, 但 ETF 價格報酬改成 r, 股票改成 r + 2%."""
    assets = Params().assets
    assets["ETF"].price_return, assets["股票"].price_return = r, r + 0.02
    return assets

axes[0].plot(rates * 100, [final_total(assets=with_price_return(r)) for r in rates], "o-")
axes[0].set(xlabel="ETF 價格報酬 % (股票 +2%, 另有股息)", ylabel="百萬 TWD", title="報酬率")

expenses = np.arange(20_000, 50_001, 5_000)
axes[1].plot(expenses / 1000, [final_total(monthly_expenses=e) for e in expenses], "o-")
axes[1].set(xlabel="每月生活開銷 (千元)", title="生活開銷")

years = range(5, 41, 5)
axes[2].plot(list(years), [final_total(years=y) for y in years], "o-")
axes[2].set(xlabel="年數", title="時間")
plt.tight_layout()

**觀察**: 生活開銷是**線性 (linear)**影響 (每月少花一塊, 就多投入一塊); 報酬率和時間則是**指數 (exponential)**影響. 這就是為什麼 "早點開始" 那麼重要.

## 7. 練習題 (exercises)

1. **翻倍時間 (doubling time)**: 解 $P(1+i)^n = 2P$ 得到 $n$, 畫出年報酬 1%–15% 下 "精確值" 和 "72 法則" 的誤差曲線. 72 法則在哪個報酬率最準?
2. **達成目標要幾年**: 起始 0 元, 每月投入 $C$, 要多久達到 1,000 萬? 從年金公式解出 $n$(提示: 取對數, take the logarithm), 再用 `simulate()` 驗證.
3. **成長年金 (growing annuity)**: 假設**每年**投入 $C$, 且投入金額每年成長 $g$(加薪), 年報酬 $r$. 推導 $n$ 年後的終值
   $$FV = C\,\frac{(1+r)^n - (1+g)^n}{r - g}$$
   (提示: 一樣用 "乘以公比再相減", 這次公比是 $\frac{1+r}{1+g}$). 當 $r = g$ 時會怎樣?
4. **通膨 (inflation)**: 通膨率 $\pi = 2\%$ 時, 實質報酬率 (real rate of return) 是 $\frac{1+r}{1+\pi} - 1$ 而不是 $r - \pi$, 為什麼? `Params` 已經有 `inflation`(目前只用來調整開銷): 讓 `simulate()` 多輸出一欄 `real_total`, 把總資產換算成 "今天的購買力"(real purchasing power), 並寫測試驗證.
5. **再平衡 (rebalancing)**: 比較 `rebalance=True` 和 `False` 30 年後的結果和資產比例. 在固定報酬率下, 哪個比較高? 為什麼? (第 1 階段加入波動後, 答案會變得更有趣.)